<a href="https://colab.research.google.com/github/pskarthikk/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row = one content page for one client on one report date.

**Time window:** The full warehouse covers 2025-01-27 through 2026-06-30. Individual clients/pages may have shorter histories because the panel is unbalanced.

In [32]:
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"

In [33]:
date_check = con.sql(f"""
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS distinct_dates
FROM {REL}
""").df()

date_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_date,last_date,distinct_dates
0,2025-01-27,2026-06-30,520


In [34]:
duplicate_grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {REL}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()

duplicate_grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-13,client_8ddc46da5414ffd8,content_1d06a2c99a935e49,2
1,2026-06-16,client_9c26c096d6e57253,content_70b02f5039a882c2,2
2,2026-06-14,client_9c26c096d6e57253,content_01ca8d0758d6614a,2
3,2026-06-14,client_1a8bf67cad4ee525,content_3084acbc2aca184b,2
4,2026-06-19,client_def0955f7a377868,content_c52c3798a1412de7,2
5,2026-06-20,client_9c26c096d6e57253,content_8d81ee3f670fcc71,2
6,2026-06-17,client_1a8bf67cad4ee525,content_dd441f559f3cb4f7,2
7,2026-06-17,client_1a8bf67cad4ee525,content_10c2a0e85769f485,2
8,2026-06-21,client_810019792c9b8efc,content_41a28b3332556857,2
9,2026-06-21,client_1a730cb2640a1abf,content_62a6c7628cfd006f,2


In [35]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", bool(HF_TOKEN))
print("Token starts with:", HF_TOKEN[:3] if HF_TOKEN else None)

Token loaded: True
Token starts with: hf_


In [36]:
from huggingface_hub import whoami

me = whoami(token=HF_TOKEN)
print("Hugging Face username:", me["name"])

Hugging Face username: pskarthikk


In [37]:
import duckdb

con = duckdb.connect()

In [38]:
con.execute(
    "CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '" + HF_TOKEN + "')"
)

In [39]:
test = con.sql("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
""").df()

test

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [40]:
print("HF token loaded:", bool(HF_TOKEN))
print("Token prefix:", HF_TOKEN[:3] if HF_TOKEN else None)

HF token loaded: True
Token prefix: hf_


In [41]:
con.sql("SELECT name, type FROM duckdb_secrets()").df()

,name,type
0,hf_token,huggingface


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Contract field classification

**Features (maximum 5):**
- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_sessions`

**Label:**
- Future performance decline proxy, derived from a later observation window.

**Context:**
- `report_date`
- `client_hash_id`
- `content_hash_id`
- `month`
- `client_has_gsc`
- `client_has_ga4`
- `gsc_data_available`
- `ga4_data_available`

**Excluded:**
- `gsc_sum_position` — redundant with the average-position signal and not needed for the feature set.
- `ga4_users` — excluded to keep the feature set limited to five features.
- `ga4_engaged_sessions` — excluded to keep the feature set limited to five features.
- `ga4_total_engagement_sec` — excluded to keep the feature set limited to five features.
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid` — excluded because these additional channel metrics are not required for the initial scoring contract.
- `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` — excluded because they are not required for the initial scoring contract.
- `scroll_events` — excluded because it is not required for the initial scoring contract.
- Any future-derived outcome information — excluded from features to prevent target leakage.

In [42]:
columns = con.sql(f"""
DESCRIBE
SELECT *
FROM {REL}
""").df()

columns[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [43]:
verification_1 = con.sql(f"""
WITH duplicate_groups AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {REL}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
)
SELECT
    (SELECT COUNT(*) FROM {REL}) AS total_rows,
    (SELECT COUNT(DISTINCT report_date) FROM {REL}) AS distinct_dates,
    (SELECT MIN(report_date) FROM {REL}) AS first_date,
    (SELECT MAX(report_date) FROM {REL}) AS last_date,
    COUNT(*) AS duplicate_groups
FROM duplicate_groups
""").df()

verification_1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_dates,first_date,last_date,duplicate_groups
0,78835655,520,2025-01-27,2026-06-30,6390


In [44]:
verification_2 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(gsc_impressions) AS missing_gsc_impressions,
    COUNT(*) - COUNT(gsc_clicks) AS missing_gsc_clicks,
    COUNT(*) - COUNT(gsc_avg_position) AS missing_gsc_avg_position,
    COUNT(*) - COUNT(ga4_pageviews) AS missing_ga4_pageviews,
    COUNT(*) - COUNT(ga4_sessions) AS missing_ga4_sessions
FROM {REL}
""").df()

verification_2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_gsc_impressions,missing_gsc_clicks,missing_gsc_avg_position,missing_ga4_pageviews,missing_ga4_sessions
0,78835655,98006,98006,49865654,29635327,29635327


In [45]:
verification_3 = con.sql(f"""
WITH history AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS observed_days,
        MIN(report_date) AS first_seen,
        MAX(report_date) AS last_seen
    FROM {REL}
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    COUNT(*) AS content_client_pairs,
    SUM(CASE WHEN observed_days >= 30 THEN 1 ELSE 0 END) AS pairs_with_30plus_days,
    SUM(CASE WHEN observed_days >= 90 THEN 1 ELSE 0 END) AS pairs_with_90plus_days,
    MIN(first_seen) AS earliest_first_seen,
    MAX(last_seen) AS latest_last_seen
FROM history
""").df()

verification_3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_client_pairs,pairs_with_30plus_days,pairs_with_90plus_days,earliest_first_seen,latest_last_seen
0,427292,407517.0,349846.0,2025-01-27,2026-06-30


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

- The panel is unbalanced: not every client-content pair is observed for the full warehouse window. Of 427,292 client-content pairs, 407,517 have at least 30 observed days and 349,846 have at least 90 observed days.
- GSC and GA4 fields are not universally available. Missingness is substantial for several proposed features, so scores may be based on different available signals across observations.
- The observed date range is 2025-01-27 through 2026-06-30; this is an observed historical window, not a guarantee of complete history for every page.
- The data supports directional, decision-support prioritization, not causal claims about why a page's performance changed.
- - The proposed grain `(report_date, client_hash_id, content_hash_id)` has 6,390 duplicate groups in the full warehouse, so consumers should not assume perfect one-row-per-key uniqueness without an additional deduplication rule.
- Future performance information must not be used as a feature when constructing the scoring model, because doing so would create target leakage.

In [46]:
data_limits_check = con.sql(f"""
WITH history AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS observed_days
    FROM {REL}
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    COUNT(*) AS total_client_content_pairs,
    SUM(CASE WHEN observed_days < 30 THEN 1 ELSE 0 END) AS pairs_under_30_days,
    SUM(CASE WHEN observed_days >= 30 THEN 1 ELSE 0 END) AS pairs_with_30plus_days,
    SUM(CASE WHEN observed_days >= 90 THEN 1 ELSE 0 END) AS pairs_with_90plus_days
FROM history
""").df()

data_limits_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_client_content_pairs,pairs_under_30_days,pairs_with_30plus_days,pairs_with_90plus_days
0,427292,19775.0,407517.0,349846.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.